# Basic data access

This notebook demonstrates the basic, top-level data-access and API-discovery methods on the `Ag3` class: exploring what releases and sample sets exist, looking up release/study/terms-of-use info for a given sample set, and introspecting the API itself.

## Set up the Ag3 data resource

In [1]:
import malariagen_data
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

## `releases`

A property (no arguments, access without parentheses) that lists the data release identifiers relevant to this `Ag3` instance, e.g. `"3.0"`, `"3.1"`, etc.

Which releases show up here is controlled entirely by arguments passed when the `Ag3` client itself was constructed, not by arguments to `releases`:
- `pre` (constructor arg): if `True`, pre-release data releases are included as well as public ones.
- `unrestricted_use_only` (constructor arg): if `True`, only releases that contain at least one sample set with unrestricted terms of use are included.
- `surveillance_use_only` (constructor arg): if `True`, only releases that contain at least one sample flagged for surveillance use are included.

Here we used the defaults for all of these, so `releases` reflects the full set of public releases available to this client.

In [2]:
ag3.releases

('3.0',
 '3.1',
 '3.2',
 '3.3',
 '3.4',
 '3.5',
 '3.6',
 '3.7',
 '3.8',
 '3.9',
 '3.10',
 '3.11',
 '3.12',
 '3.13',
 '3.14',
 '3.15',
 '3.16')

## `sample_sets`

Returns a dataframe of sample sets, one row per sample set, with columns `sample_set`, `sample_count`, `study_id`, `study_url`, `terms_of_use_expiry_date`, `terms_of_use_url`, `release`, and `unrestricted_use`.

Parameters:
- `release` (optional): a single release identifier (e.g. `"3.0"`) or a sequence of release identifiers to restrict the returned sample sets to those releases. If `None` (the default, used below), sample sets from every relevant release are returned. This is the natural way to survey everything available, and also gives us real `sample_set` identifiers to use as arguments for the lookup methods below (e.g. `"AG1000G-BF-A"`).

In [3]:
df_sample_sets = ag3.sample_sets()
df_sample_sets

,sample_set,sample_count,study_id,study_url,terms_of_use_expiry_date,terms_of_use_url,release,unrestricted_use
0,AG1000G-AO,81,AG1000G-AO,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
1,AG1000G-BF-A,181,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
2,AG1000G-BF-B,102,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
3,AG1000G-BF-C,13,AG1000G-BF-2,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
4,AG1000G-CD,76,AG1000G-CD,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,3.0,True
...,...,...,...,...,...,...,...,...
96,1323-VO-GM-NGWA-VMF00235,188,1323-VO-GM-NGWA,https://www.malariagen.net/partner_study/1323-...,2026-04-09,https://malariagen.github.io/vector-data/ag3/a...,3.9,True
97,1323-VO-GM-NGWA-VMF00242,1630,1323-VO-GM-NGWA,https://www.malariagen.net/partner_study/1323-...,2026-04-09,https://malariagen.github.io/vector-data/ag3/a...,3.9,True
98,1329-VO-GA-CHRISTOPHE-VMF00228,146,1329-VO-GA-CHRISTOPHE,https://www.malariagen.net/partner_study/1329-...,2026-04-09,https://malariagen.github.io/vector-data/ag3/a...,3.9,True
99,bergey-2019,113,bergey-2019,https://doi.org/10.1111/eva.12878,NaN,https://onlinelibrary.wiley.com/doi/10.1111/ev...,3.9,True


## `lookup_release`

Given a sample set identifier, returns the string identifier of the release that sample set belongs to.

Parameters:
- `sample_set` (required): the sample set identifier to look up, e.g. one of the values seen in the `sample_set` column above. Changing this to a different sample set changes which release is returned; an unknown sample set raises a `ValueError`.

In [4]:
ag3.lookup_release(sample_set="AG1000G-BF-A")

'3.0'

## `lookup_study`

Given a sample set identifier, returns the study identifier (`study_id`) of the study that generated it.

Parameters:
- `sample_set` (required): the sample set identifier to look up. Different sample sets from the same contributing study will return the same study identifier; sample sets from different studies return different identifiers.

In [5]:
ag3.lookup_study(sample_set="AG1000G-BF-A")

'AG1000G-BF-1'

## `describe_api`

Lists all public API methods available on this `Ag3` instance, one row per method, with columns `method`, `summary` (first line of the docstring) and `category`.

Parameters:
- `category` (optional): restrict the listing to one of `"data"`, `"analysis"`, or `"plot"`. `None` (the default) returns all methods regardless of category. Here we use `category="data"` (a non-default value) to show only the data-access methods, which is a more illustrative example than dumping the entire API at once — an invalid category string raises a `ValueError`.

In [6]:
ag3.describe_api(category="data")

,method,summary,category
0,aim_calls,"Access ancestry informative marker SNP sites, ...",data
1,aim_metadata,Access ancestry-informative marker (AIM) metad...,data
2,aim_variants,Access ancestry informative marker variants.,data
3,cnv_coverage_calls,Access CNV HMM data from genome-wide CNV disco...,data
4,cnv_discordant_read_calls,Access CNV discordant read calls data.,data
5,cnv_hmm,Access CNV HMM data from CNV calling.,data
6,cohorts_metadata,Access cohort membership metadata for one or m...,data
7,gene_cnv,"Compute modal copy number by gene, from HMM data.",data
8,gene_cnv_frequencies,"Compute modal copy number by gene, then comput...",data
9,gene_cnv_frequencies_advanced,"Group samples by taxon, area (space) and perio...",data


## `v3_wild`

A legacy convenience property (no arguments) that returns a plain Python list of sample set identifiers from the `"3.0"` release, excluding the `AG1000G-X` sample set (which contains laboratory crosses rather than wild-caught mosquitoes). Useful as a shorthand `sample_sets` argument to other methods when you want "all the wild Ag1000G phase 3 samples".

In [7]:
ag3.v3_wild

['AG1000G-AO',
 'AG1000G-BF-A',
 'AG1000G-BF-B',
 'AG1000G-BF-C',
 'AG1000G-CD',
 'AG1000G-CF',
 'AG1000G-CI',
 'AG1000G-CM-A',
 'AG1000G-CM-B',
 'AG1000G-CM-C',
 'AG1000G-FR',
 'AG1000G-GA-A',
 'AG1000G-GH',
 'AG1000G-GM-A',
 'AG1000G-GM-B',
 'AG1000G-GM-C',
 'AG1000G-GN-A',
 'AG1000G-GN-B',
 'AG1000G-GQ',
 'AG1000G-GW',
 'AG1000G-KE',
 'AG1000G-ML-A',
 'AG1000G-ML-B',
 'AG1000G-MW',
 'AG1000G-MZ',
 'AG1000G-TZ',
 'AG1000G-UG']

## `lookup_study_info`

Given a sample set identifier, returns a dict with more detail about the study than `lookup_study` alone, namely `study_id` and `study_url`.

Parameters:
- `sample_set` (required): the sample set identifier to look up. As with `lookup_study`, changing this changes which study's info is returned.

In [8]:
ag3.lookup_study_info(sample_set="AG1000G-BF-A")

{'study_id': 'AG1000G-BF-1',
 'study_url': 'https://www.malariagen.net/partner_study/AG1000G-BF-1'}

## `lookup_terms_of_use_info`

Given a sample set identifier, returns a dict describing the terms of use that apply to that sample set: `terms_of_use_expiry_date`, `terms_of_use_url`, and `unrestricted_use` (whether the data can currently be used without restriction, e.g. because the terms-of-use period has expired).

Parameters:
- `sample_set` (required): the sample set identifier to look up. Different sample sets can have different terms-of-use expiry dates and URLs, since they may come from different contributing studies.

In [9]:
ag3.lookup_terms_of_use_info(sample_set="AG1000G-BF-A")

{'terms_of_use_expiry_date': '2025-01-01',
 'terms_of_use_url': 'https://www.malariagen.net/data/our-approach-sharing-data/ag1000g-terms-of-use/',
 'unrestricted_use': True}